In [15]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cluster-test")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "pyspark-notebook")
    .getOrCreate()
)

df = spark.range(10)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



check spark version

In [20]:
spark.version

'3.4.0'

Download taxi data

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -P /data

Create spark session and read the parquet file

In [2]:
spark = ( SparkSession.builder .appName("yellow taxi dataframe") .master("spark://spark-master:7077") .config("spark.driver.host", "pyspark-notebook") .getOrCreate() )

In [5]:
df = spark.read.parquet("/data/yellow_tripdata_2025-11.parquet") 
df.printSchema() 
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

create table to run sql queries

In [6]:
df.registerTempTable("taxi_table")

/usr/local/spark/python/pyspark/sql/dataframe.py:330: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


SQL queries

In [7]:


spark.sql("""
SELECT count(1) FROM taxi_table
WHERE CAST(tpep_pickup_datetime AS DATE) = '2025-11-15'
""").show()



+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [8]:
spark.sql("""
SELECT max(tpep_dropoff_datetime - tpep_pickup_datetime) FROM taxi_table
""").show()

+---------------------------------------------------+
|max((tpep_dropoff_datetime - tpep_pickup_datetime))|
+---------------------------------------------------+
|                               INTERVAL '3 18:38...|
+---------------------------------------------------+



create and force schema

In [10]:
from pyspark.sql import types

In [11]:


df_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True),
    types.StructField("Airport_fee", types.DoubleType(), True),
    types.StructField("cbd_congestion_fee", types.DoubleType(), True),
])



In [12]:
yellow_df = spark.read.schema(df_schema).parquet("/data/yellow_tripdata_2025-11.parquet")
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

add zone table (all steps)

In [13]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-06 18:49:15--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.163, 18.239.38.181, 18.239.38.147, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-06 18:49:15 (157 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [18]:
zones_df = spark.read.option("header", "true").csv("/data/taxi_zone_lookup.csv")
zones_df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [19]:
zones_df.registerTempTable("zones_table")

Last question of hw

In [32]:
spark.sql(
    """
    WITH etc AS (
      SELECT PULocationID, count(PULocationID) as zone_count from taxi_table
      group by PULocationID
    ) 
    SELECT z.Zone, etc.zone_count FROM etc
    INNER JOIN zones_table as z
    on etc.PULocationID=Z.LocationID
    ORDER BY etc.zone_count
    """
).show()

+--------------------+----------+
|                Zone|zone_count|
+--------------------+----------+
|Governor's Island...|         1|
|       Arden Heights|         1|
|Eltingville/Annad...|         1|
|       Port Richmond|         3|
|   Rossville/Woodrow|         4|
|       Rikers Island|         4|
| Green-Wood Cemetery|         4|
|         Great Kills|         4|
|         Jamaica Bay|         5|
|         Westerleigh|        12|
|        Crotona Park|        14|
|             Oakwood|        14|
|New Dorp/Midland ...|        14|
|       West Brighton|        14|
|       Willets Point|        15|
|Breezy Point/Fort...|        16|
|Saint George/New ...|        17|
|       Broad Channel|        18|
|     Mariners Harbor|        21|
|Heartland Village...|        22|
+--------------------+----------+
only showing top 20 rows



In [29]:
spark.sql(
    """
    SELECT * FROM zones_table
    where ZONE LIKE '%Governor%'
    """
).show(truncate=False)

+----------+---------+---------------------------------------------+------------+
|LocationID|Borough  |Zone                                         |service_zone|
+----------+---------+---------------------------------------------+------------+
|103       |Manhattan|Governor's Island/Ellis Island/Liberty Island|Yellow Zone |
|104       |Manhattan|Governor's Island/Ellis Island/Liberty Island|Yellow Zone |
|105       |Manhattan|Governor's Island/Ellis Island/Liberty Island|Yellow Zone |
+----------+---------+---------------------------------------------+------------+

